# NIFTY 50 strategy research: walkthrough

This notebook is a guided tour of the results. All logic lives in the tested `niftyquant` package
(`src/niftyquant/`), and `python -m niftyquant` regenerates every table and figure in `results/` and `figures/`.

**Question:** after realistic costs and an honest benchmark, do classic equity signals or an ML ranker add
value in large-cap Indian equities?

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
from niftyquant.config import Config
from niftyquant.pipeline import Research
from niftyquant import metrics as mx
pd.set_option("display.float_format", "{:.3f}".format)
cfg = Config()
R = Research(cfg)
print(R.prices.shape, R.prices.index[0].date(), "->", R.prices.index[-1].date())

(2708, 50) 2015-01-02 -> 2025-12-31


## 1. Data quality
Yahoo's adjusted prices miss two demergers. Left untreated they show up as fake -39% and -40% one-day losses.
`data.adjust_demergers` back-adjusts them; the check below should list only genuine large moves.

In [2]:
from niftyquant import data
data.quality_report(R.prices)

,Date,Ticker,return
0,2019-10-29,TMPV.NS,0.359


## 2. Strategy performance, net of 15 bps one-way costs
Signals use information up to the close of day *t* and execute at the close of *t+1*.

In [3]:
res, perf = R.performance()
perf[["CAGR", "Gross CAGR", "Sharpe", "Sharpe 95% CI low", "Sharpe 95% CI high", "Max Drawdown", "Calmar",
      "Beta", "Alpha (ann.)", "Alpha t-stat", "Excess CAGR vs EW", "Turnover (ann.)"]]

,CAGR,Gross CAGR,Sharpe,Sharpe 95% CI low,Sharpe 95% CI high,Max Drawdown,Calmar,Beta,Alpha (ann.),Alpha t-stat,Excess CAGR vs EW,Turnover (ann.)
Strategy,,,,,,,,,,,,
Equal Weight,0.234,0.235,1.008,0.276,1.794,-0.375,0.624,0.956,0.092,5.392,0.000,0.766
Momentum 12-1,0.239,0.251,0.852,0.162,1.575,-0.449,0.533,1.029,0.100,2.455,0.006,6.478
Low Volatility,0.171,0.176,0.791,0.144,1.487,-0.269,0.633,0.699,0.055,2.221,-0.063,3.282
ST Reversal,0.196,0.277,0.682,0.016,1.408,-0.438,0.448,1.060,0.062,1.608,-0.038,43.459
Trend-Filtered EW,0.181,0.185,0.947,0.305,1.597,-0.138,1.311,0.501,0.077,2.650,-0.053,2.591
MACD+RSI (v1),-0.005,-0.003,-0.775,-1.476,-0.132,-0.366,-0.014,-0.365,-0.032,-2.000,-0.239,1.212
NIFTY 50 index,0.129,NaN,0.473,-0.179,1.189,-0.384,0.337,1.000,NaN,NaN,-0.104,NaN


## 3. The benchmark problem: survivorship bias
An equal-weight portfolio of *today's* NIFTY 50 members beats the index by ~10 pp a year. Much of that gap
comes from picking the stocks with hindsight. That is why the strategies are judged against **Equal Weight**
(same universe, no skill), not against the index.

In [4]:
R.survivorship_gap(res["Equal Weight"].returns)

{"EW of today's constituents CAGR": np.float64(0.23374059668126757),
 'NIFTY 50 price index CAGR': np.float64(0.12946148660459356),
 'Gap (pp per year)': np.float64(0.10427911007667401)}

## 4. Costs and robustness

In [5]:
grid = R.cost_grid()
display(grid.style.format("{:.1%}"))
R.breakeven_costs(grid)

,Equal Weight,Momentum 12-1,Low Volatility,ST Reversal,Trend-Filtered EW,MACD+RSI (v1)
cost_bps,,,,,,
0.000000,23.5%,25.1%,17.6%,27.7%,18.5%,-0.3%
5.000000,23.5%,24.7%,17.4%,24.9%,18.4%,-0.4%
10.000000,23.4%,24.3%,17.2%,22.2%,18.2%,-0.4%
15.000000,23.4%,23.9%,17.1%,19.6%,18.1%,-0.5%
25.000000,23.3%,23.1%,16.7%,14.5%,17.8%,-0.6%
40.000000,23.1%,21.9%,16.1%,7.3%,17.3%,-0.8%
60.000000,22.9%,20.4%,15.3%,-1.7%,16.7%,-1.0%


Momentum 12-1       22.933
Low Volatility       0.000
ST Reversal          7.775
Trend-Filtered EW    0.000
MACD+RSI (v1)        0.000
Name: Break-even cost vs EW (bps), dtype: float64

In [6]:
R.momentum_grid()

,Top 10%,Top 20%,Top 30%,Top 50%
Formation period,,,,
3-1 months,0.672,0.756,0.823,0.875
6-1 months,0.631,0.825,0.808,0.908
9-1 months,0.768,0.955,1.047,1.043
12-1 months,0.694,0.852,0.943,0.998


In [7]:
R.subperiods(res).pivot(index="Strategy", columns="Period", values="Sharpe")

Period,Development 2016-2021,Hold-out 2022-2025
Strategy,,
Equal Weight,1.036,0.975
Low Volatility,0.879,0.638
MACD+RSI (v1),-0.938,-0.594
Momentum 12-1,0.772,1.000
NIFTY 50 index,0.511,0.409
ST Reversal,0.835,0.399
Trend-Filtered EW,1.009,0.850


## 5. Walk-forward machine learning
Each year is predicted by models trained only on earlier data, with a 21-day purge gap. The target is
cross-sectional: will a stock beat the median stock over the next month?

In [8]:
scores, ic, ml_table, _ = R.machine_learning()
display(scores.pivot(index="Year", columns="Model", values="AUC"))
display(ic)
ml_table[["CAGR", "Sharpe", "Max Drawdown", "Excess CAGR vs EW", "Turnover (ann.)"]]

Model,Gradient Boosting,Logistic
Year,,
2018,0.506,0.482
2019,0.520,0.527
2020,0.525,0.507
2021,0.501,0.532
2022,0.483,0.489
2023,0.506,0.553
2024,0.489,0.489
2025,0.520,0.488


,Feature,Mean IC,t-stat
5,vol_63,0.041,1.995
8,dist_sma50,-0.033,-1.623
1,ret_21,-0.032,-1.691
6,rsi_14,-0.032,-1.681
3,mom_12_1,0.025,1.135
4,vol_21,0.022,1.137
10,max_ret_21,0.017,1.030
2,ret_63,-0.015,-0.710
0,ret_5,-0.014,-0.782
11,volume_z,-0.012,-0.768


,CAGR,Sharpe,Max Drawdown,Excess CAGR vs EW,Turnover (ann.)
Strategy,,,,,
ML Ranker,0.230,0.787,-0.396,0.012,14.521
Equal Weight,0.219,0.897,-0.375,0.000,0.807
Momentum 12-1,0.210,0.720,-0.449,-0.009,6.646


## 6. Conclusions
* No signal beats the naive equal-weight portfolio on a risk-adjusted basis once costs are included. All Sharpe
  confidence intervals overlap.
* Short-term reversal has the largest gross edge, but at ~44x annual turnover it is gone by ~8 bps of costs.
* Momentum is robust across parameters but only adds about 0.5 pp of CAGR, with more risk.
* The trend filter is a risk-management tool: it cuts max drawdown from -38% to -14% and doubles the Calmar
  ratio, at the cost of ~5 pp of CAGR. Most of that evidence comes from a single crash (March 2020).
* The ML ranker's AUC is ~0.51 out of sample: no reliable edge from price/volume features alone.

See `docs/research_note.md` for the full write-up.